# Predicting Trametinib Sensitivity in Cancer Cell Lines

## Introduction

This project investigates whether cancer driver mutation profiles can be used to predict the sensitivity of cancer cell lines to **Trametinib**, a small-molecule inhibitor of MEK1 and MEK2 in the RAS–RAF–MEK–ERK (MAPK) signalling pathway.

Drug-response data from the **Genomics of Drug Sensitivity in Cancer (GDSC)** are integrated with driver mutation data from **Cell Model Passports** using Sanger Model IDs.

The primary objective is to explore relationships between genomic alterations and experimentally measured Trametinib response (`LN_IC50`) and, subsequently, to develop machine learning models that predict drug sensitivity from genomic features.

### Research Question

**Can cancer driver mutation profiles predict cancer cell line sensitivity to Trametinib?**

### Initial Objectives

1. Explore and validate the GDSC drug-response data.
2. Explore the cancer driver mutation data.
3. Identify cancer cell lines with both Trametinib response and mutation data.
4. Investigate the prevalence of mutations in MAPK pathway and related cancer genes.
5. Construct a machine-learning-ready dataset linking genomic features to Trametinib response.

## Notebook Contents

This notebook focuses on the initial exploration and preparation of the project data.

The analysis includes:

1. Loading the GDSC drug-response dataset.
2. Inspecting dataset dimensions, variables, and missing values.
3. Exploring the distribution of drug-response measurements.
4. Identifying Trametinib-treated cancer cell lines.
5. Loading and exploring the Cell Model Passports driver mutation dataset.
6. Assessing overlap between Trametinib-treated cell lines and available mutation data.
7. Exploring mutation prevalence in MAPK pathway and related cancer genes.
8. Preparing the data for subsequent feature engineering and machine learning analysis.

No predictive models are trained in this notebook. The purpose is to understand the available data and establish a clean foundation for the next stages of the project.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
gdsc = pd.read_csv("../data/GDSC2-dataset.csv")
mutations = pd.read_csv("../data/mutations_summary.csv")

## Cancer Driver Mutations Data

The mutations dataset contatains details of mutations in known cancer driver genes in cancer cell lines. Initial exploration is performed to understand dataset structure and quality.

In [10]:
mutations.shape

(13699, 19)

In [11]:
mutations.head()

,gene_symbol,ensembl_gene_id,transcript_id,model_name,model_id,protein_mutation,rna_mutation,cdna_mutation,chromosome,position,reference,alternative,cancer_driver,cancer_predisposition_variant,effect,vaf,coding,source,gene_id
0,CASP8,ENSG00000064012,CCDS42798,GR-ST,SIDM01259,p.F338fs*11,r.1207delU,c.1011delT,chr2,201284846.0,AT,A,t,f,frameshift,0.4921,t,Sanger,SIDG03513
1,BMPR2,ENSG00000204217,CCDS33361,GR-ST,SIDM01259,p.N343fs*14,r.2172delA,c.1024delA,chr2,202530849.0,GA,G,t,f,frameshift,0.5156,t,Sanger,SIDG02364
2,MCM3AP,ENSG00000160294,CCDS13734,GR-ST,SIDM01259,p.?,r.5363+1delg,c.5038+1delg,chr21,46244805.0,AC,A,t,f,ess_splice,0.5227,t,Sanger,SIDG17421
3,DGCR8,ENSG00000128191,CCDS13773,GR-ST,SIDM01259,p.P727fs*18,r.2595_2596insa,c.2175_2176insA,chr22,20108940.0,C,CA,t,f,frameshift,0.4931,t,Sanger,SIDG06490
4,MB21D2,ENSG00000180611,CCDS3302,GR-ST,SIDM01259,p.N216fs*11,r.663delA,c.647delA,chr3,192799214.0,AT,A,t,f,frameshift,0.5747,t,Sanger,SIDG17353


In [12]:
mutations.info()

<class 'pandas.DataFrame'>
RangeIndex: 13699 entries, 0 to 13698
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   gene_symbol                    13699 non-null  str    
 1   ensembl_gene_id                13699 non-null  str    
 2   transcript_id                  13301 non-null  str    
 3   model_name                     13699 non-null  str    
 4   model_id                       13699 non-null  str    
 5   protein_mutation               13699 non-null  str    
 6   rna_mutation                   13699 non-null  str    
 7   cdna_mutation                  13699 non-null  str    
 8   chromosome                     13301 non-null  str    
 9   position                       13301 non-null  float64
 10  reference                      13301 non-null  str    
 11  alternative                    13301 non-null  str    
 12  cancer_driver                  13699 non-null  str    
 1

In [13]:
mutations.columns

Index(['gene_symbol', 'ensembl_gene_id', 'transcript_id', 'model_name',
       'model_id', 'protein_mutation', 'rna_mutation', 'cdna_mutation',
       'chromosome', 'position', 'reference', 'alternative', 'cancer_driver',
       'cancer_predisposition_variant', 'effect', 'vaf', 'coding', 'source',
       'gene_id'],
      dtype='str')

In [14]:
mutations.isnull().sum().sort_values(ascending=False)

position                         398
alternative                      398
transcript_id                    398
chromosome                       398
reference                        398
source                             0
coding                             0
vaf                                0
effect                             0
cancer_predisposition_variant      0
cancer_driver                      0
gene_symbol                        0
ensembl_gene_id                    0
cdna_mutation                      0
rna_mutation                       0
protein_mutation                   0
model_id                           0
model_name                         0
gene_id                            0
dtype: int64

In [15]:
print("Unique cell lines:", mutations["model_id"].nunique())
print("Unique genes:", mutations["gene_symbol"].nunique())
print("Unique mutation effects:", mutations["effect"].nunique())
print("Unique data sources:", mutations["source"].nunique())
print("Cancer driver categories:", mutations["cancer_driver"].nunique())


Unique cell lines: 1533
Unique genes: 613
Unique mutation effects: 9
Unique data sources: 2
Cancer driver categories: 2


In [16]:
mutations["effect"].value_counts()

effect
frameshift            4905
missense              4536
nonsense              2657
ess_splice            1451
inframe                 80
start_lost              47
stop_lost               21
5prime_UTR_variant       1
3prime_UTR_variant       1
Name: count, dtype: int64

In [17]:
mutations["source"].value_counts()

source
Sanger    11603
Broad      2096
Name: count, dtype: int64

## GDSC Drug Sensitivity Data

The GDSC dataset contains experimentally measured responses of cancer cell lines to anti-cancer compounds. Initial exploration is performed to understand the dataset structure, available variables, and data quality.

In [3]:
print("Dataset shape:", gdsc.shape)

gdsc.head()

Dataset shape: (242036, 19)


,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,COSMIC_ID,CELL_LINE_NAME,SANGER_MODEL_ID,TCGA_DESC,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,COMPANY_ID,WEBRELEASE,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE
0,GDSC2,343,15946310,683667,PFSK-1,SIDM01132,MB,1003,Camptothecin,TOP1,DNA replication,1046,Y,0.0001,0.1,-1.463887,0.930220,0.089052,0.433123
1,GDSC2,343,15946548,684052,A673,SIDM00848,UNCLASSIFIED,1003,Camptothecin,TOP1,DNA replication,1046,Y,0.0001,0.1,-4.869455,0.614970,0.111351,-1.421100
2,GDSC2,343,15946830,684057,ES5,SIDM00263,UNCLASSIFIED,1003,Camptothecin,TOP1,DNA replication,1046,Y,0.0001,0.1,-3.360586,0.791072,0.142855,-0.599569
3,GDSC2,343,15947087,684059,ES7,SIDM00269,UNCLASSIFIED,1003,Camptothecin,TOP1,DNA replication,1046,Y,0.0001,0.1,-5.044940,0.592660,0.135539,-1.516647
4,GDSC2,343,15947369,684062,EW-11,SIDM00203,UNCLASSIFIED,1003,Camptothecin,TOP1,DNA replication,1046,Y,0.0001,0.1,-3.741991,0.734047,0.128059,-0.807232


In [4]:
gdsc.info()

<class 'pandas.DataFrame'>
RangeIndex: 242036 entries, 0 to 242035
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   DATASET          242036 non-null  str    
 1   NLME_RESULT_ID   242036 non-null  int64  
 2   NLME_CURVE_ID    242036 non-null  int64  
 3   COSMIC_ID        242036 non-null  int64  
 4   CELL_LINE_NAME   242036 non-null  str    
 5   SANGER_MODEL_ID  242036 non-null  str    
 6   TCGA_DESC        240969 non-null  str    
 7   DRUG_ID          242036 non-null  int64  
 8   DRUG_NAME        242036 non-null  str    
 9   PUTATIVE_TARGET  214881 non-null  str    
 10  PATHWAY_NAME     242036 non-null  str    
 11  COMPANY_ID       242036 non-null  int64  
 12  WEBRELEASE       242036 non-null  str    
 13  MIN_CONC         242036 non-null  float64
 14  MAX_CONC         242036 non-null  float64
 15  LN_IC50          242036 non-null  float64
 16  AUC              242036 non-null  float64
 17  RM

In [5]:
gdsc.describe()

,NLME_RESULT_ID,NLME_CURVE_ID,COSMIC_ID,DRUG_ID,COMPANY_ID,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE
count,242036.0,2.420360e+05,2.420360e+05,242036.000000,242036.000000,242036.000000,242036.000000,242036.000000,242036.000000,242036.000000,2.420360e+05
mean,343.0,1.606806e+07,9.921059e+05,1594.042444,1042.966604,0.023143,23.462279,2.817079,0.882592,0.082779,7.312962e-10
std,0.0,7.028749e+04,2.209819e+05,398.740714,16.911327,0.158738,158.622810,2.762229,0.146998,0.042695,9.993925e-01
min,343.0,1.594631e+07,6.836670e+05,1003.000000,1001.000000,0.000010,0.010000,-8.747724,0.006282,0.003274,-8.254501e+00
25%,343.0,1.600719e+07,9.068050e+05,1149.000000,1043.000000,0.003002,3.000000,1.508018,0.849449,0.051107,-6.568485e-01
50%,343.0,1.606807e+07,9.097200e+05,1631.000000,1046.000000,0.010005,10.000000,3.236731,0.944196,0.076083,1.058000e-02
75%,343.0,1.612893e+07,1.240144e+06,1912.000000,1046.000000,0.010005,10.000000,4.700110,0.974934,0.106105,6.560362e-01
max,343.0,1.618978e+07,1.789883e+06,2499.000000,1101.000000,2.001054,2000.000000,13.820189,0.998904,0.299984,7.978776e+00


In [6]:
print("Unique cell lines:", gdsc["SANGER_MODEL_ID"].nunique())
print("Unique drugs:", gdsc["DRUG_NAME"].nunique())
print("Unique cancer types:", gdsc["TCGA_DESC"].nunique())

Unique cell lines: 969
Unique drugs: 286
Unique cancer types: 32


In [7]:
gdsc.isnull().sum().sort_values(ascending=False)

PUTATIVE_TARGET    27155
TCGA_DESC           1067
PATHWAY_NAME           0
RMSE                   0
AUC                    0
LN_IC50                0
MAX_CONC               0
MIN_CONC               0
WEBRELEASE             0
COMPANY_ID             0
DATASET                0
NLME_RESULT_ID         0
DRUG_NAME              0
DRUG_ID                0
SANGER_MODEL_ID        0
CELL_LINE_NAME         0
COSMIC_ID              0
NLME_CURVE_ID          0
Z_SCORE                0
dtype: int64

In [8]:
print("Duplicate rows:", gdsc.duplicated().sum())

Duplicate rows: 0


In [9]:
drug_summary = (
    gdsc.groupby(["DRUG_NAME", "PUTATIVE_TARGET", "PATHWAY_NAME"])
        .agg(
            n_cell_lines=("SANGER_MODEL_ID", "nunique"),
            mean_ln_ic50=("LN_IC50", "mean"),
            std_ln_ic50=("LN_IC50", "std")
        )
        .sort_values("n_cell_lines", ascending=False)
)

drug_summary.head(30)

,,,n_cell_lines,mean_ln_ic50,std_ln_ic50
DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,,,
MG-132,"Proteasome, CAPN1",Protein stability and degradation,969,-1.283393,0.722336
5-Fluorouracil,Antimetabolite (DNA & RNA),Other,968,4.386510,1.715815
Palbociclib,"CDK4, CDK6",Cell cycle,968,3.440316,1.570024
Docetaxel,Microtubule stabiliser,Mitosis,968,-3.415391,2.548620
MK-2206,"AKT1, AKT2",PI3K/MTOR signaling,968,2.724173,1.486104
Camptothecin,TOP1,DNA replication,968,-2.259385,1.836654
PD0325901,"MEK1, MEK2",ERK MAPK signaling,968,1.342255,2.143469
Nutlin-3a (-),MDM2,p53 pathway,968,4.416219,1.787005
Staurosporine,Broad spectrum kinase inhibitor,RTK signaling,968,-2.793135,1.384601


### Selection of Trametinib

Trametinib was selected as the initial compound for further analysis. It is a small-molecule inhibitor of MEK1 and MEK2 within the MAPK signalling pathway.

The dataset contains Trametinib response measurements for 966 unique cancer cell lines and shows substantial variation in LN_IC50 values. This provides a sufficiently large and heterogeneous dataset for investigating whether genomic features are associated with differences in drug response.

The MAPK pathway also provides a biologically interpretable framework for investigating genomic alterations in genes such as BRAF, KRAS, NRAS, NF1, MAP2K1, and MAP2K2.